<a href="https://colab.research.google.com/github/Leo278V/Final-Assignment-PDS/blob/Final-Assignment-V2/Final-Assignment-V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Rule Based Approach

In [18]:
import pandas as pd
import numpy as np


In [8]:
# Data Path CSV Files

data_path_department = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/department-v2.csv?token=GHSAT0AAAAAADQCRM6W33FX6JAZTU3E4GNC2KJBWIQ"
df_department = pd.read_csv(data_path_department)
df_department.head()

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management


In [9]:
data_path_seniority = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/seniority-v2.csv?token=GHSAT0AAAAAADQCRM6X7NSLPC6XU2WSLJAU2KJBX6A"
df_seniortiy = pd.read_csv(data_path_seniority)
df_seniortiy.head()

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior


In [17]:
# Data Path Annotated LinkedIN Profiles
data_path_profiles = "https://raw.githubusercontent.com/Leo278V/Final-Assignment-PDS/refs/heads/Final-Assignment-V2/linkedin_experience_annotated.csv?token=GHSAT0AAAAAADQCRM6WQ2VITJLKQHBMMK442KJCA7A"
df_profiles = pd.read_csv(data_path_profiles)
df_profiles.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [32]:
# Filtering for active
df_active = df_profiles[df_profiles["status"] == "ACTIVE"].copy()

df_active.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,NaN,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,NaN,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,NaN,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,NaN,ACTIVE,Other,Management,0


In [45]:
# Creating Dicitionaries

#Seniority
seniority_dict = (
    df_seniortiy
    .dropna(subset=["text", "label"])
    .assign(keyword=lambda x: x["text"].str.lower())
    .set_index("keyword")["label"]
    .to_dict()
)

#Department
department_dict = (
    df_department
    .dropna(subset=["text", "label"])
    .assign(keyword=lambda x: x["text"].str.lower())
    .set_index("keyword")["label"]
    .to_dict()
)
print(seniority_dict)
print(department_dict)

{'analyst': 'Junior', 'analyste financier': 'Junior', 'anwendungstechnischer mitarbeiter': 'Junior', 'application engineer': 'Senior', 'applications engineer': 'Senior', 'architecte si - chef de projet applicatif': 'Lead', 'associate': 'Junior', 'associate - research': 'Junior', 'associate partner': 'Junior', 'associate recruiter': 'Junior', 'associated partner': 'Junior', 'bi analyst': 'Junior', 'call-center-agent- service - mitarbeiter': 'Junior', 'chargée de communication / chef de projet événementiel': 'Lead', 'chef de marché résidences services senior - solutions logicielles métier': 'Lead', 'chef de projet': 'Lead', 'chef de projet communication et marketing': 'Lead', 'chef de projet erp': 'Lead', 'chef de projet évènementiel': 'Lead', 'chef de projet événementiel / event coordinator': 'Lead', 'chef de projet marketing': 'Lead', 'chef de projet marketing digital': 'Lead', 'chef de projet marketing digital senior': 'Lead', 'chef de projet marketing et communication': 'Lead', 'chef

In [35]:
# Feature Text

TEXT_COLS = ["position", "organization"]

df_active["text"] = (
    df_active[TEXT_COLS]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
)




In [36]:
# Matching Dictionaries

def dict_match(text, label_dict):
    for keyword, label in label_dict.items():
        if keyword in text:
            return label
    return np.nan


In [37]:
# Application on Annotated CVs

df_active["pred_department"] = df_active["text"].apply(
    lambda x: dict_match(x, department_dict)
)

df_active["pred_seniority"] = df_active["text"].apply(
    lambda x: dict_match(x, seniority_dict)
)


In [38]:
#Successful Classification
dept_coverage = df_active["pred_department"].notna().mean()
sen_coverage = df_active["pred_seniority"].notna().mean()

print(f"Department Coverage: {dept_coverage:.2%}")
print(f"Seniority Coverage: {sen_coverage:.2%}")


Department Coverage: 40.93%
Seniority Coverage: 62.12%


In [40]:
#Accuracy with assigned labels for Department
dept_eval = df_active.dropna(
    subset=["department", "pred_department"]
)

dept_accuracy = (
    dept_eval["department"] == dept_eval["pred_department"]
).mean()

print(f"Department Accuracy: {dept_accuracy:.2%}")

Department Accuracy: 39.61%


In [43]:
#Accuracy with assigned labels for Department
sen_eval = df_active.dropna(
    subset=["seniority", "pred_seniority"]
)

sen_accuracy = (
    sen_eval["seniority"] == sen_eval["pred_seniority"]
).mean()

print(f"Seniority Accuracy: {sen_accuracy:.2%}")

Seniority Accuracy: 57.11%
